# ☀️ Solar Filament Segmentation Challenge 2026
##  Hybrid Pipeline: YOLO (Bounding Box Detection) + SAM 2.1 (Precision Segmentation)

### 📌 Executive Summary & Strategy
In this notebook, our objective is to segment solar filaments with pixel-level precision on $2048 \times 2048$ resolution H-Alpha solar imagery from the **MAGFiLO 1.0** dataset.

Instead of relying solely on an end-to-end segmentation network, we adopt a **Two-Stage Hybrid Pipeline**:
1. **Stage 1 (Object Detection - YOLO):** Locate solar filaments across high-resolution full-disk imagery to generate bounding box prompts.
2. **Stage 2 (Refinement Segmentation - SAM 2.1):** Pass YOLO's bounding boxes as prompts into Segment Anything Model 2.1 (SAM 2.1) to extract zero-shot, fine-grained filament boundaries and delicate structures.

---

##  1. Exploratory Data Analysis (EDA)
In this section, we inspect dataset image counts, evaluate class distributions, and visualize ground-truth segmentation masks overlaid directly on solar disk imagery.

In [ ]:
import os
import glob
import json
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

In [ ]:
DATASET_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"

if not os.path.exists(DATASET_DIR):

    DATASET_DIR = "MAGFiLO_1.0_Kaggle_2026"

TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train", "train_images")
TEST_IMG_DIR = os.path.join(DATASET_DIR, "test", "test_images")
JSON_PATH = os.path.join(DATASET_DIR, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")

In [ ]:
train_images = glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpeg")) + glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg"))
test_images = glob.glob(os.path.join(TEST_IMG_DIR, "*.jpeg")) + glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))

coco = COCO(JSON_PATH)

cat_ids = coco.getCatIds()
cats = coco.loadCats(cat_ids)
categories_df = pd.DataFrame(cats)

# Count total annotations per class
cat_counts = {}
for cat in cats:
    ann_ids = coco.getAnnIds(catIds=cat['id'])
    cat_counts[cat['name']] = len(ann_ids)

categories_df['annotation_count'] = categories_df['name'].map(cat_counts)

print("=" * 60)
print("📊 MAGFiLO 1.0 DATASET SUMMARY")
print("=" * 60)
print(f"📸 Total Training Images : {len(train_images)}")
print(f"🧪 Total Test Images     : {len(test_images)}")
print(f"🏷️ Total Annotated Masks : {len(coco.anns)}")
print("=" * 60)
print("\n📂 CLASS DISTRIBUTION:")
print(categories_df[['id', 'name', 'supercategory', 'annotation_count']].to_string(index=False))
print("=" * 60)

In [ ]:
plt.figure(figsize=(8, 4))
bars = plt.bar(categories_df['name'], categories_df['annotation_count'], color=['#3498db', '#2ecc71', '#e74c3c', '#f1c40f'])
plt.title("Filament Mask Counts by Class", fontsize=12, fontweight='bold')
plt.xlabel("Class Name")
plt.ylabel("Mask Count")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 10, f"{yval}", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# 3. GROUND-TRUTH MASK VISUALIZATION
# ---------------------------------------------------------
def visualize_sample_annotations(coco_obj, img_dir, num_samples=3):
    img_ids = list(coco_obj.imgs.keys())
    sample_img_ids = random.sample(img_ids, num_samples)

    fig, axes = plt.subplots(1, num_samples, figsize=(6 * num_samples, 6))
    if num_samples == 1:
        axes = [axes]

    for idx, img_id in enumerate(sample_img_ids):
        img_info = coco_obj.imgs[img_id]
        img_path = os.path.join(img_dir, img_info['file_name'])

        image = cv2.imread(img_path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        overlay = image.copy()
        ann_ids = coco_obj.getAnnIds(imgIds=[img_id])
        anns = coco_obj.loadAnns(ann_ids)

        np.random.seed(42)
        for ann in anns:
            seg = ann['segmentation']
            color = np.random.randint(50, 255, size=3).tolist()

            if isinstance(seg, list):  # Polygon format
                for poly in seg:
                    pts = np.array(poly).reshape(-1, 2).astype(np.int32)
                    cv2.fillPoly(overlay, [pts], color)
                    cv2.polylines(image, [pts], isClosed=True, color=(255, 255, 255), thickness=2)
            else:  # RLE format
                rle = maskUtils.frPyObjects(seg, h, w)
                mask = maskUtils.decode(rle)
                if len(mask.shape) == 3:
                    mask = mask[:, :, 0]
                overlay[mask == 1] = color

       
        combined_img = cv2.addWeighted(overlay, 0.4, image, 0.6, 0)

        axes[idx].imshow(combined_img)
        axes[idx].set_title(f"Image: {img_info['file_name']}\nMask Count: {len(anns)}", fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()


visualize_sample_annotations(coco, TRAIN_IMG_DIR, num_samples=3)

In [ ]:
def visualize_9_samples_grid(coco_obj, img_dir, grid_size=3, fig_size=(15, 15)):
    img_ids = list(coco_obj.imgs.keys())
    num_samples = grid_size * grid_size
    
    selected_img_ids = random.sample(img_ids, min(num_samples, len(img_ids)))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=fig_size)
    
    for idx, img_id in enumerate(selected_img_ids):
        row = idx // grid_size
        col = idx % grid_size
        ax = axes[row, col]

        img_info = coco_obj.imgs[img_id]
        img_path = os.path.join(img_dir, img_info['file_name'])

        image = cv2.imread(img_path)
        if image is None:
            ax.axis('off')
            continue
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape
        overlay = image.copy()

        ann_ids = coco_obj.getAnnIds(imgIds=[img_id])
        anns = coco_obj.loadAnns(ann_ids)

        np.random.seed(idx + 42)  # Reproducible random colors per image
        for ann in anns:
            seg = ann['segmentation']
            color = np.random.randint(50, 255, size=3).tolist()

            if isinstance(seg, list):  # Polygon format
                for poly in seg:
                    pts = np.array(poly).reshape(-1, 2).astype(np.int32)
                    cv2.fillPoly(overlay, [pts], color)
                    cv2.polylines(image, [pts], isClosed=True, color=(255, 255, 255), thickness=1)
            else:  # RLE format
                rle = maskUtils.frPyObjects(seg, h, w)
                mask = maskUtils.decode(rle)
                if len(mask.shape) == 3:
                    mask = mask[:, :, 0]
                overlay[mask == 1] = color

        combined_img = cv2.addWeighted(overlay, 0.4, image, 0.6, 0)

        ax.imshow(combined_img)
        ax.set_title(f"{img_info['file_name']}\nMasks: {len(anns)}", fontsize=8)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

visualize_9_samples_grid(coco, TRAIN_IMG_DIR, grid_size=3, fig_size=(15, 15))

##  2. Model Training: YOLO Detection Architecture

### 💡 Model Selection & Architectural Rationale (`YOLOv5x6u`)
To accurately locate delicate solar filaments across $2048 \times 2048$ full-disk imagery, we selected the **`YOLOv5x6u`** architecture:
* **High-Resolution Optimization (P6 Architecture):** The `x6` variant features an extended P6 feature pyramid network explicitly designed for ultra-high-resolution images ($2048 \times 2048$). It prevents spatial information loss and excels at detecting small, long, and narrow filament structures.
* **Focused Object Detection:** Dedicated solely to object localization, YOLO avoids the trade-offs of single-stage instance segmentation, maximizing bounding box recall and spatial precision.

---

### 🧩 Decoupled Hybrid Strategy (Zero-Shot SAM 2.1)
* **Untrained / Off-the-Shelf SAM 2.1:** The SAM 2.1 model is **not fine-tuned or trained**. It is utilized directly off-the-shelf in a zero-shot fashion, receiving YOLO's predicted bounding boxes as prompt inputs.
* **Performance Synergy:** Decoupling detection from mask extraction allows each model to specialize in its core strength—YOLO for fast, reliable object discovery and SAM 2.1 for crisp, pixel-perfect boundary extraction.

---

### ⚙️ Hardware & Training Configuration
Due to the heavy computational demand of training at full $2048 \times 2048$ resolution, training was executed on an external high-performance node:

| Parameter | Configuration / Value |
| :--- | :--- |
| **Compute Infrastructure** | NVIDIA A100 GPU |
| **Input Resolution (`imgsz`)** | $2048 \times 2048$ (Native Resolution) |
| **Total Epochs** | 70 Epochs |
| **Batch Size** | 4 |
| **Peak VRAM Allocation** | ~34 GB |

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov5x6u.pt") 

results = model.train(
    data="...data.yaml",
    epochs=70,
    imgsz=2048,
    batch=4 
)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

In [ ]:
TRAIN_OUTPUT_DIR = "/kaggle/input/datasets/berkutayasan/solar-filament-segmentation-challenge/train-2"

if not os.path.exists(TRAIN_OUTPUT_DIR):
    TRAIN_OUTPUT_DIR = "/content/runs/detect/train-2"

def load_rgb_image(file_name):
    path = os.path.join(TRAIN_OUTPUT_DIR, file_name)
    if os.path.exists(path):
        img = cv2.imread(path)
        if img is not None:
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return None

results_img = load_rgb_image("results.png")

if results_img is not None:
    plt.figure(figsize=(16, 7))
    plt.imshow(results_img)
    plt.title("YOLOv5x6u Training Progress & Loss Curves (results.png)", fontsize=14, fontweight='bold', pad=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️")

In [ ]:
curve_files = [
    ("BoxF1_curve.png", "Box F1-Confidence Curve"),
    ("BoxP_curve.png", "Box Precision-Confidence Curve"),
    ("BoxR_curve.png", "Box Recall-Confidence Curve"),
    ("BoxPR_curve.png", "Box Precision-Recall Curve")
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (file_name, title) in enumerate(curve_files):
    img_rgb = load_rgb_image(file_name)
    
    if img_rgb is not None:
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(title, fontsize=12, fontweight='bold', pad=8)
    else:
        axes[idx].text(0.5, 0.5, f"⚠️ {file_name}\nBulunamadı", ha='center', va='center', fontsize=11)
    axes[idx].axis('off')

plt.suptitle("YOLO Performance Evaluation Curves (2x2)", fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

cm_img = load_rgb_image("confusion_matrix_normalized.png")

if cm_img is None:
    cm_img = load_rgb_image("confusion_matrix.png")

if cm_img is not None:
    plt.figure(figsize=(10, 10))
    plt.imshow(cm_img)
    plt.title("Normalized Confusion Matrix", fontsize=14, fontweight='bold', pad=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️")


##  3. End-to-End Test Inference Pipeline: YOLO Detection + SAM 2.1 Segmentation

### 🔄 Two-Stage Hybrid Workflow
With our Stage 1 detector (`YOLOv5x6u`) trained, we execute the complete inference pipeline across $2048 \times 2048$ resolution test images using zero-shot **SAM 2.1 Large** for precision mask extraction:

1. **Bounding Box Detection & NMS Filtering:**
   * **Confidence Threshold (`conf=0.4`):** Filters out low-confidence background noise and spurious detections.
   * **Non-Maximum Suppression (`iou=0.3`):** Resolves overlapping duplicate bounding boxes by retaining only the highest-scoring candidate.
2. **Priority Sorting:**
   * Bounding boxes are sorted by YOLO confidence scores in descending order ($S_1 > S_2 > \dots > S_N$) to ensure dominant, high-confidence filaments take priority during mask generation.
3. **SAM 2.1 Prompted Segmentation:**
   * Sorted bounding boxes are passed as spatial prompts into SAM 2.1 (`sam2.1_l.pt`), generating pixel-accurate zero-shot binary masks.

---

### 🛠️ Post-Processing & Mask Refinement

To ensure compliance with competition evaluation criteria and maximize segmentation score, three post-processing steps are applied sequentially:

| Step | Technique | Objective / Function |
| :--- | :--- | :--- |
| **1. Boundary Closing** | Morphological Closing ($5 \times 5$ Elliptical Kernel) | Reconnects fragmented fine filaments ("barbs") and seals internal boundary gaps. |
| **2. Zero-Overlap Policy** | Occupied Pixel Matrix ($2048 \times 2048$) | Ensures strict zero-pixel overlap by reserving pixels for higher-confidence masks first. |
| **3. Noise Filtering** | Minimum Area Threshold ($< 30$ pixels) | Eliminates isolated micro-artifacts and false-positive pixel clusters. |

---

### 📦 Final Submission Format
Refined binary masks are encoded into **COCO Run-Length Encoding (RLE)** strings and saved to `submission.csv` with the required schema (`filament_id`, `segmentation_rle`).

In [ ]:
from ultralytics import YOLO
from ultralytics import SAM

# Load the YOLO model
yolo_model = YOLO("/kaggle/input/datasets/berkutayasan/solar-filament-segmentation-challenge/train-2/weights/best.pt")  # pretrained YOLO model

# Run batched inference on a list of images
results = yolo_model("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20111208173814Mh.jpeg")  # return a list of Results objects

# Load the SAM model
sam_model = SAM("sam2.1_l.pt")
for result in results:
     class_ids = result.boxes.cls.int().tolist()  # noqa
     if len(class_ids):
         boxes = result.boxes.xyxy  # Boxes object for bbox outputs
         sam_results = sam_model(result.orig_img, bboxes=boxes, verbose=False, save=True, device=0, show_labels=False, show_conf=False,batch=1,conf=0.7)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO, SAM

yolo_model = YOLO("/kaggle/input/datasets/berkutayasan/solar-filament-segmentation-challenge/train-2/weights/best.pt")
sam_model = SAM("sam2.1_l.pt")

img_path = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20111015063134Lh.jpeg"

results = yolo_model(
    img_path,
    conf=0.4,   
    iou=0.3,   
    imgsz=2048,
    verbose=False
)

for result in results:
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy  

        sam_results = sam_model(
            result.orig_img,
            bboxes=boxes,
            verbose=False,
            device=0
        )

        annotated_img = sam_results[0].plot(
            line_width=1,
        )

        annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(12, 12))
        plt.imshow(annotated_img_rgb)
        plt.title(f"YOLO + SAM 2.1 Result | Predicted Filament Number: {len(boxes)}", fontsize=12)
        plt.axis('off')
        plt.show()

    else:
        print("No flament")

In [ ]:
import os
import glob
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO, SAM

YOLO_MODEL_PATH = "/kaggle/input/datasets/berkutayasan/solar-filament-segmentation-challenge/train-2/weights/best.pt"
SAM_MODEL_PATH = "sam2.1_l.pt"
TEST_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"


if not os.path.exists(TEST_DIR):
    TEST_DIR = "/kaggle/input/magfilo-1-0-kaggle-2026/test/test_images"

yolo_model = YOLO(YOLO_MODEL_PATH)
sam_model = SAM(SAM_MODEL_PATH)

test_image_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpeg"))) + sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")))

if len(test_image_paths) == 0:
    raise FileNotFoundError(f"No images found in test directory: {TEST_DIR}")

random.seed(42)  # Fixed seed for reproducibility
num_samples = min(20, len(test_image_paths))
selected_img_paths = random.sample(test_image_paths, num_samples)

fig, axes = plt.subplots(5, 4, figsize=(20, 24))
axes = axes.flatten()

print(f"Processing and laying out a total of {num_samples} images into a 5x4 grid...")

for idx in range(20):
    if idx < num_samples:
        img_path = selected_img_paths[idx]
        file_name = os.path.basename(img_path)
        
        # YOLO Detection (conf=0.4, iou=0.3)
        yolo_results = yolo_model(
            img_path,
            conf=0.4,
            iou=0.3,
            imgsz=2048,
            verbose=False
        )[0]
        
        if yolo_results.boxes is not None and len(yolo_results.boxes) > 0:
            boxes = yolo_results.boxes.xyxy
            
            # SAM 2.1 Prompted Segmentation
            sam_results = sam_model(
                yolo_results.orig_img,
                bboxes=boxes,
                verbose=False,
                device=0
            )[0]
            
            
            annotated_img = sam_results.plot(
                line_width=1,
            )
            annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
            
            axes[idx].imshow(annotated_img_rgb)
            axes[idx].set_title(f"{file_name}\nFilaments: {len(boxes)}", fontsize=9, fontweight='bold')
        else:
            # Render original image for samples without detections
            orig_img_rgb = cv2.cvtColor(yolo_results.orig_img, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(orig_img_rgb)
            axes[idx].set_title(f"{file_name}\nNo Detections", fontsize=9, fontweight='bold', color='red')
    
    axes[idx].axis('off')

plt.suptitle("YOLOv5x6u + SAM 2.1 Test Inference Grid (20 Samples)", fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



### 🔬 Literary Contribution & Baseline Findings
This study was conducted as a novel exploratory approach to contribute to solar image processing literature by evaluating a fully **decoupled two-stage pipeline** for H-Alpha solar filament segmentation. 

By separating **bounding box localization (YOLOv5x6u)** from **pixel-level mask generation (SAM 2.1)**, we established a modular framework that leverages zero-shot foundation models without requiring end-to-end instance segmentation training. Using SAM 2.1 completely **off-the-shelf (without fine-tuning)**, this hybrid strategy achieved a competition baseline score of **0.30**.

---

### 🚀 Key Takeaways & Potential Improvements
While the zero-shot decoupled pipeline proves that off-the-shelf foundation models can effectively segment complex solar features from box prompts alone, several clear avenues exist to significantly boost performance beyond the 0.30 baseline:

1. **Domain-Specific SAM Fine-Tuning:** Fine-tuning SAM 2.1's lightweight mask decoder on H-Alpha solar imagery to better adapt its attention to low-contrast filament edges and delicate "barbs."
2. **Multi-Prompting & Point Strategy:** Enhancing box prompts with positive/negative point prompts derived from YOLO feature maps to guide SAM 2.1 in ambiguous solar regions.
3. **Advanced Prompt Filtering & Ensembling:** Lowering detection confidence thresholds while employing multi-model box ensembling to ensure near-100% recall before mask generation.